# 04 · Cascading impacts on hospitals and water supply (3D)

**Question for this notebook:** when grid outages from notebooks 01 and 03 cut power to substations, how far do the failures spread into water supply and hospitals, and how many critical service-hours are lost?

| Step | Method / data | Main reference |
|---|---|---|
| Facilities | Hospitals and water-supply nodes in Nagpur from **OpenStreetMap** (Overpass API) | — |
| Population | **WorldPop** 2020 100 m population via **Earth Engine**, summed over each water-supply zone | Gorelick et al. (2017) |
| Facility → bus | **synthetic** mapping of facilities to load buses of the test network (weighted by bus load) | stated limitation |
| Hospital → water | each hospital is served by its nearest water-supply node (real geography) | — |
| Dependencies | NetworkX directed graph: bus → pump → hospital / water zone, processed in topological order | Rinaldi et al. (2001); Ouyang (2014) |
| Buffers | service-reservoir storage, hospital water tanks, diesel generator fuel | Thacker et al. (2017); Pant et al. (2018) |
| Metric | **critical service-hours lost** = hospital-hours down + water-zone-hours without supply; person-hours without water; split into direct (power) and cascade (water) | Sendai Target D |

**Run order:** after 01 (and 03 for heat scenarios).

**Outputs** (in `results/04_cascade/`): `facilities.csv`, `cascade_params.json`, `dependency_graph.graphml`, `cascade_results.csv`, `cascade_summary.csv`, `facility_outages.csv`, figures.

> **The facility-to-bus mapping is synthetic.** The test network is not Nagpur's grid. Real facility locations and population are used only to give the dependency layer realistic size and structure.

## 0. Setup

In [ ]:
!pip install -q earthengine-api shapely

In [ ]:
import os, sys, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

# ---- Shared project settings (identical in all five notebooks) ----
PROJECT_ID = "goggle-earth-499404"          # Google Cloud project registered for Earth Engine
SITE = "Nagpur, Maharashtra"
LAT, LON, ELEV_M = 21.15, 79.09, 310
TZ = "Asia/Kolkata"

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True                             # keep results in Google Drive so the notebooks can share them
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/grid_stress_nagpur"
else:
    ROOT = os.path.abspath("grid_stress_nagpur")

CACHE = os.path.join(ROOT, "cache")
RES = {k: os.path.join(ROOT, "results", k) for k in
       ["01_contingency", "02_feeder", "03_heat", "04_cascade", "05_mitigation"]}
for d in [CACHE, *RES.values()]:
    os.makedirs(d, exist_ok=True)
OUT = RES["04_cascade"]

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")
    plt.show()

def require(path, notebook):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run notebook {notebook} first (with USE_DRIVE = True).")
    return path

print("Project folder:", ROOT)

## 1. Parameters
Buffer values are assumptions to state and vary:
- **Hospital feeders.** Indian utilities generally exempt hospitals on dedicated/express feeders from rotational load shedding, so a hospital loses grid supply only when its bus is (almost) fully cut (`HOSP_SHED_THRESHOLD`). Pumping stations are not exempt: they lose supply once half the bus load is shed.
- **Water storage.** Elevated service reservoirs (ESRs) typically hold several hours of zone demand. Hospitals keep their own overhead and underground tanks.
- **Diesel generators.** Large hospitals usually have DG sets with about a day of fuel; small hospitals and nursing homes may have a few hours or none.

In [ ]:
BBOX = (20.98, 78.95, 21.25, 79.22)    # south, west, north, east: Nagpur city and surroundings
SEED = 7

P = dict(
    HOSP_SHED_THRESHOLD=0.95,   # hospital grid supply lost if >= 95% of its bus load is shed
    PUMP_SHED_THRESHOLD=0.50,   # pumping station lost if >= 50% of its bus load is shed
    WATER_STORAGE_H=8,          # service-reservoir storage, hours of zone demand
    REFILL_PER_H=0.5,           # storage refilled per hour once pumps run again (hours of demand per hour)
    HOSP_WATER_H=12,            # hospital on-site water storage, hours
    DG_HOURS_MAJOR=24,          # diesel fuel autonomy, major hospitals, hours
    DG_HOURS_SMALL=4,           # diesel fuel autonomy, small hospitals, hours
    PUMP_DG_H=0,                # diesel backup at pumping stations, hours
)
BEDS_DEFAULT = {"major": 200, "small": 30}
MAJOR_PATTERN = r"medical college|gmc|igmc|aiims|government|govt|district|civil|mayo|institute|super|esic|railway|cancer|memorial"
TAIL_H = 24                     # hours simulated after the grid is restored (storage refill)
FALLBACK = dict(n_hospitals=80, n_water=15, population=2.9e6)   # used only if OSM / Earth Engine are unreachable

## 2. Facilities from OpenStreetMap
Two Overpass queries inside the bounding box:
- **Hospitals:** `amenity=hospital` or `healthcare=hospital`.
- **Water supply:** `man_made=water_works` and `man_made=pumping_station` (pumping), plus `man_made=water_tower` (elevated reservoirs, each filled by pumping). Each node defines a supply zone.

Ways and relations are reduced to their centre point. Results are cached so the analysis is reproducible.

In [ ]:
import requests, re

OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.private.coffee/api/interpreter",
            "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter"]
HEADERS = {"User-Agent": "grid-stress-nagpur/1.0 (research notebook; Colab)", "Accept": "application/json"}

def overpass(query, cache_name, tries_per_mirror=2):
    # Query Overpass with several mirrors and retries; cache only non-empty answers
    f = os.path.join(CACHE, cache_name)
    if os.path.exists(f):
        data = json.load(open(f))
        if data.get("elements"):
            return data
    for url in OVERPASS:
        for k in range(tries_per_mirror):
            try:
                r = requests.post(url, data={"data": query}, headers=HEADERS, timeout=300)
                if r.status_code != 200:
                    print(f"Overpass {url}: HTTP {r.status_code} {r.text[:120]!r}")
                    time.sleep(15 * (k + 1)); continue
                data = r.json()
                if not data.get("elements"):
                    print(f"Overpass {url}: empty answer {data.get('remark', '')[:120]}")
                    time.sleep(15 * (k + 1)); continue
                json.dump(data, open(f, "w"))
                print(f"Overpass {url}: {len(data['elements'])} elements")
                return data
            except Exception as e:
                print(f"Overpass {url} failed: {str(e)[:120]}")
                time.sleep(10 * (k + 1))
    return None

def osm_points(data, kind):
    rows = []
    for el in (data or {}).get("elements", []):
        lat = el.get("lat", el.get("center", {}).get("lat")); lon = el.get("lon", el.get("center", {}).get("lon"))
        if lat is None:
            continue
        t = el.get("tags", {})
        rows.append(dict(osm_id=f"{el['type']}/{el['id']}", lat=lat, lon=lon, name=t.get("name", ""),
                         beds=pd.to_numeric(t.get("beds"), errors="coerce"),
                         subtype=t.get("man_made", t.get("amenity", t.get("healthcare", ""))), kind=kind))
    df = pd.DataFrame(rows)
    if len(df):                                         # same facility mapped as node and building
        df["key"] = df.name.str.lower().str.strip() + "_" + df.lat.round(3).astype(str) + "_" + df.lon.round(3).astype(str)
        df = df.drop_duplicates("key").drop(columns="key")
    return df

s, w, n, e = BBOX
qh = f'[out:json][timeout:120];(nwr["amenity"="hospital"]({s},{w},{n},{e});nwr["healthcare"="hospital"]({s},{w},{n},{e}););out center tags;'
qw = f'[out:json][timeout:120];(nwr["man_made"="water_works"]({s},{w},{n},{e});nwr["man_made"="pumping_station"]({s},{w},{n},{e});nwr["man_made"="water_tower"]({s},{w},{n},{e}););out center tags;'
hosp_osm = osm_points(overpass(qh, "osm_hospitals_nagpur.json"), "hospital")
water_osm = osm_points(overpass(qw, "osm_water_nagpur.json"), "water")
SOURCE = {"hospitals": "OpenStreetMap", "water": "OpenStreetMap"}

rng = np.random.default_rng(SEED)
def synthetic_points(nn, kind):
    return pd.DataFrame(dict(osm_id=[f"synthetic/{kind}{i}" for i in range(nn)],
                             lat=rng.uniform(s, n, nn), lon=rng.uniform(w, e, nn), name="", beds=np.nan,
                             subtype="synthetic", kind=kind))
if len(hosp_osm) < 10:
    print("*** Re-run this cell (Overpass is often busy) before accepting synthetic locations ***")
    print("WARNING: too few hospitals from OSM, using synthetic locations"); hosp_osm = synthetic_points(FALLBACK["n_hospitals"], "hospital"); SOURCE["hospitals"] = "synthetic"
if len(water_osm) < 5:
    print("WARNING: too few water nodes from OSM, using synthetic locations"); water_osm = synthetic_points(FALLBACK["n_water"], "water"); SOURCE["water"] = "synthetic"
print(f"Hospitals: {len(hosp_osm)} ({SOURCE['hospitals']}); water-supply nodes: {len(water_osm)} ({SOURCE['water']})")
print(water_osm.subtype.value_counts().to_dict())

### 2.1 Hospital tiers
A hospital is **major** if OSM lists ≥ 100 beds or its name matches a public / tertiary-care pattern (medical college, government, district, institute, …); otherwise it is **small**. Missing bed counts get a tier default.

In [ ]:
hosp = hosp_osm.copy()
hosp["tier"] = np.where((hosp.beds >= 100) | hosp.name.str.lower().str.contains(MAJOR_PATTERN, regex=True), "major", "small")
hosp["beds"] = hosp.beds.fillna(hosp.tier.map(BEDS_DEFAULT))
hosp["id"] = [f"H{i:03d}" for i in range(len(hosp))]
water = water_osm.copy()
water["id"] = [f"W{i:02d}" for i in range(len(water))]
hosp.groupby("tier").agg(count=("id", "size"), beds=("beds", "sum"))

## 3. Water-supply zones and population (WorldPop via Earth Engine)
Each water node serves its Voronoi cell (the area closer to it than to any other node), clipped to the bounding box. Zone population = sum of WorldPop 2020 (100 m, constrained) over the cell. If Earth Engine is unavailable, the fallback spreads `FALLBACK['population']` by zone area.

In [ ]:
from shapely.geometry import MultiPoint, Point, box
from shapely.ops import voronoi_diagram

env = box(w, s, e, n)
pts = MultiPoint([(lo, la) for la, lo in zip(water.lat, water.lon)])
cells = [c.intersection(env) for c in voronoi_diagram(pts, envelope=env).geoms]
zone_poly = {}
for wid, la, lo in zip(water.id, water.lat, water.lon):
    zone_poly[wid] = next(c for c in cells if c.buffer(1e-9).contains(Point(lo, la)))

POP_SOURCE = "WorldPop 2020 (Earth Engine)"
try:
    import ee
    try:
        ee.Initialize(project=PROJECT_ID)
    except Exception:
        ee.Authenticate(); ee.Initialize(project=PROJECT_ID)
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Polygon([list(map(list, p.exterior.coords))]), {"id": wid})
                               for wid, p in zone_poly.items()])
    pop_img = (ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.eq("country", "IND"))
               .filter(ee.Filter.eq("year", 2020)).mosaic().select("population"))
    res = pop_img.reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=100).getInfo()
    pop = {f["properties"]["id"]: f["properties"].get("sum", 0.0) for f in res["features"]}
except Exception as ex:
    print("Earth Engine unavailable (", str(ex)[:80], ") - using fallback population")
    POP_SOURCE = "fallback (area-weighted)"
    areas = pd.Series({k: v.area for k, v in zone_poly.items()})
    pop = (FALLBACK["population"] * areas / areas.sum()).to_dict()
water["pop"] = water.id.map(pop).fillna(0.0)
print(f"Population in study area: {water['pop'].sum():,.0f} ({POP_SOURCE})")

## 4. Link facilities to the grid and to water
- **Hospital → water node:** nearest by great-circle distance (real geography).
- **Facility → load bus (synthetic):** each facility is assigned to a load bus of the test network with probability proportional to bus load, a proxy for how urban the bus is. The seed is fixed so the mapping is reproducible. Change `SEED` to test how sensitive the results are to the mapping.

In [ ]:
buses = pd.read_csv(require(os.path.join(RES["01_contingency"], "network_buses.csv"), "01"), index_col="bus")
load_buses = buses[buses.load_mw > 0]

def haversine(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, (la1, lo1, la2, lo2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

d = haversine(hosp.lat.values[:, None], hosp.lon.values[:, None], water.lat.values[None, :], water.lon.values[None, :])
hosp["pump"] = water.id.values[d.argmin(1)]
hosp["dist_to_water_km"] = d.min(1)

rng = np.random.default_rng(SEED)
pw = (load_buses.load_mw / load_buses.load_mw.sum()).values
hosp["bus"] = rng.choice(load_buses.index.values, size=len(hosp), p=pw)
water["bus"] = rng.choice(load_buses.index.values, size=len(water), p=pw)

fac = pd.concat([
    hosp.assign(type="hospital", pop=0.0)[["id", "type", "tier", "beds", "name", "lat", "lon", "bus", "pump", "pop", "osm_id"]],
    water.assign(type="pump", tier="", beds=0.0, pump="")[["id", "type", "tier", "beds", "name", "lat", "lon", "bus", "pump", "pop", "osm_id"]],
], ignore_index=True)
fac["mapping"] = "synthetic bus assignment"
fac.to_csv(os.path.join(OUT, "facilities.csv"), index=False)
print(f"{(fac.type == 'hospital').sum()} hospitals and {(fac.type == 'pump').sum()} water nodes on {fac.bus.nunique()} of {len(load_buses)} load buses")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for wid, p in zone_poly.items():
    xs, ys = p.exterior.xy; ax.fill(xs, ys, alpha=0.12, color="#2c6fbb", lw=0.6, ec="#2c6fbb")
ax.scatter(water.lon, water.lat, marker="^", s=40, color="#2c6fbb", label="water-supply node", zorder=3)
for t, c, sz in [("small", "#e08e0b", 12), ("major", "#c0392b", 40)]:
    h = hosp[hosp.tier == t]; ax.scatter(h.lon, h.lat, s=sz, color=c, label=f"hospital ({t})", zorder=4)
for _, h in hosp.iterrows():
    wv = water.set_index("id").loc[h.pump]; ax.plot([h.lon, wv.lon], [h.lat, wv.lat], color="#999999", lw=0.3, zorder=1)
ax.set_xlim(w, e); ax.set_ylim(s, n); ax.set_aspect(1 / np.cos(np.radians(LAT)))
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.legend(frameon=False, fontsize=7, loc="lower left")
ax.set_title(f"Nagpur: hospitals, water-supply zones ({SOURCE['hospitals']}, {POP_SOURCE.split(' (')[0]})")
savefig("fig01_facilities_map.png")

## 5. Dependency graph (NetworkX)
Nodes: load buses, pumping stations, water zones and hospitals. A directed edge \(u \to v\) means *v depends on u*:
- bus → pump (power), bus → hospital (power)
- pump → water zone (water), pump → hospital (water)

The graph must be acyclic. Its **topological generations** give the order in which states are computed: buses first, then pumps, then zones and hospitals. Adding another sector (telecom, wastewater, …) only needs more node types and edges.

**State rules (hourly):**
- *Pump powered* = bus supplied (shed fraction below the pump threshold), or backup available.
- *Zone has water* = pump powered, or the service reservoir is not yet empty. The reservoir drains 1 hour of demand per hour without pumping and refills at `REFILL_PER_H` once pumping resumes.
- *Hospital powered* = bus supplied (hospital threshold), or PV + battery (notebook 05), or diesel while fuel lasts.
- *Hospital has water* = zone has water, or its own tank is not yet empty.
- *Hospital functional* = powered **and** has water. Hours lost while powered but without water are **cascade** hours.

In [ ]:
# ---- Cascade model: power -> water -> hospitals (shared by notebooks 04 and 05) ----
import networkx as nx


def build_dependency_graph(fac):
    """Directed dependency graph. Edge u -> v means 'v needs u'.
    bus -> pump (power), bus -> hospital (power), pump -> hospital (water), pump -> water zone (water)."""
    G = nx.DiGraph()
    for b in sorted(fac.bus.unique()):
        G.add_node(f"bus{b}", kind="bus", bus=int(b), layer=0)
    for _, f in fac[fac.type == "pump"].iterrows():
        G.add_node(f["id"], kind="pump", layer=1)
        G.add_edge(f"bus{f['bus']}", f["id"], dep="power")
        G.add_node(f"zone_{f['id']}", kind="water_zone", pop=float(f["pop"]), layer=2)
        G.add_edge(f["id"], f"zone_{f['id']}", dep="water")
    for _, f in fac[fac.type == "hospital"].iterrows():
        G.add_node(f["id"], kind="hospital", tier=f["tier"], beds=float(f["beds"]), layer=2)
        G.add_edge(f"bus{f['bus']}", f["id"], dep="power")
        G.add_edge(f["pump"], f["id"], dep="water")
    assert nx.is_directed_acyclic_graph(G)
    return G


def dependency_arrays(G):
    """Turn the graph into index arrays, processed layer by layer (topological generations)."""
    nodes = {k: [n for n, d in G.nodes(data=True) if d["kind"] == k] for k in ["bus", "pump", "hospital", "water_zone"]}
    pos = {k: {n: i for i, n in enumerate(v)} for k, v in nodes.items()}
    def pred(n, kind):
        return next(u for u in G.predecessors(n) if G.nodes[u]["kind"] == kind)
    arr = dict(
        nodes=nodes,
        bus_ids=np.array([G.nodes[n]["bus"] for n in nodes["bus"]]),
        pump_bus=np.array([pos["bus"][pred(p, "bus")] for p in nodes["pump"]]),
        hosp_bus=np.array([pos["bus"][pred(h, "bus")] for h in nodes["hospital"]]),
        hosp_pump=np.array([pos["pump"][pred(h, "pump")] for h in nodes["hospital"]]),
        zone_pump=np.array([pos["pump"][pred(z, "pump")] for z in nodes["water_zone"]]),
        zone_pop=np.array([G.nodes[z]["pop"] for z in nodes["water_zone"]]),
        hosp_beds=np.array([G.nodes[h]["beds"] for h in nodes["hospital"]]),
        hosp_tier=np.array([G.nodes[h]["tier"] for h in nodes["hospital"]]),
        generations=[sorted({G.nodes[n]["kind"] for n in g}) for g in nx.topological_generations(G)],
    )
    return arr


def shed_fraction_matrix(ts_scenario, bus_ids, bus_load, horizon):
    """Hourly fraction of each bus's load that is shed (T x n_bus)."""
    F = np.zeros((horizon, len(bus_ids)))
    if len(ts_scenario):
        piv = ts_scenario.pivot_table(index="hour", columns="bus", values="shed_mw", aggfunc="sum")
        piv = piv.reindex(index=range(horizon), columns=bus_ids).fillna(0.0)
        F = np.clip(piv.values / np.maximum(bus_load.reindex(bus_ids).values, 1e-9), 0, 1)
    return F


def _battery_step(on_grid, soc, cap, pv_kw, load_kw, charge_rate):
    """One hour of PV + battery. Returns (supplied, new_soc). Grid-connected: battery recharges."""
    surplus = pv_kw - load_kw
    soc_off = soc + np.where(surplus >= 0, np.minimum(surplus, cap - soc), surplus)
    ok_off = soc_off >= -1e-9
    soc_off = np.clip(soc_off, 0, cap)
    soc_on = np.minimum(cap, soc + charge_rate * cap)
    return np.where(on_grid, True, ok_off), np.where(on_grid, soc_on, soc_off)


def cascade_simulate(A, F, P, backup=None):
    """Hourly simulation of facility states for one scenario.

    A: dependency arrays; F: shed fraction (T x n_bus); P: parameters;
    backup: optional {"hospital"|"pump": dict(pv_kwp, batt_kwh, load_kw (T x n), pv (T,))}.
    Returns per-facility hourly states."""
    T = F.shape[0]
    npump, nh = len(A["pump_bus"]), len(A["hosp_bus"])
    pump_grid = F[:, A["pump_bus"]] < P["PUMP_SHED_THRESHOLD"]
    hosp_grid = F[:, A["hosp_bus"]] < P["HOSP_SHED_THRESHOLD"]

    def run_backup(kind, grid):
        if not backup or kind not in backup:
            return np.zeros_like(grid)
        b = backup[kind]
        cap = b["batt_kwh"].astype(float)
        soc = cap * P.get("SOC_START", 1.0)
        sup = np.zeros_like(grid)
        for t in range(T):
            pvk = b["pv_kwp"] * b["pv"][t]
            s, soc = _battery_step(grid[t], soc, cap, pvk, b["load_kw"][t], P.get("CHARGE_RATE", 0.5))
            sup[t] = s & ~grid[t]
        return sup

    pump_backup = run_backup("pump", pump_grid)
    pfuel = np.full(npump, float(P.get("PUMP_DG_H", 0)))

    # water network storage (elevated service reservoirs), hours of demand
    S = np.full(npump, float(P["WATER_STORAGE_H"]))
    pump_on = np.zeros((T, npump), bool)
    zone_water = np.zeros((T, npump), bool)
    for t in range(T):
        g = pump_grid[t] | pump_backup[t]
        dg = ~g & (pfuel > 0)
        pfuel = np.where(dg, pfuel - 1, pfuel)
        on = g | dg
        pump_on[t] = on
        zone_water[t] = on | (S > 0)
        S = np.where(on, np.minimum(P["WATER_STORAGE_H"], S + P["REFILL_PER_H"]), np.maximum(S - 1, 0))

    hosp_backup = run_backup("hospital", hosp_grid)
    fuel = np.where(A["hosp_tier"] == "major", P["DG_HOURS_MAJOR"], P["DG_HOURS_SMALL"]).astype(float)
    W = np.full(nh, float(P["HOSP_WATER_H"]))
    hp, hw = np.zeros((T, nh), bool), np.zeros((T, nh), bool)
    for t in range(T):
        g = hosp_grid[t] | hosp_backup[t]
        dg = ~g & (fuel > 0)
        fuel = np.where(dg, fuel - 1, fuel)
        hp[t] = g | dg
        zw = zone_water[t, A["hosp_pump"]]
        hw[t] = zw | (W > 0)
        W = np.where(zw, np.minimum(P["HOSP_WATER_H"], W + P["REFILL_PER_H"]), np.maximum(W - 1, 0))
    return dict(pump_grid=pump_grid, pump_on=pump_on, zone_water=zone_water,
                hosp_grid=hosp_grid, hosp_power=hp, hosp_water=hw, hosp_ok=hp & hw)


def cascade_metrics(A, R):
    down = ~R["hosp_ok"]
    power_down = ~R["hosp_power"]
    water_only = down & ~power_down
    zone_down = ~R["zone_water"]
    return dict(
        hospitals_grid_off=int((~R["hosp_grid"]).any(0).sum()),
        hospitals_affected=int(down.any(0).sum()),
        hospital_hours_lost=float(down.sum()),
        hospital_hours_power=float((down & power_down).sum()),
        hospital_hours_water=float(water_only.sum()),
        bed_hours_lost=float((down * A["hosp_beds"]).sum()),
        pumps_grid_off=int((~R["pump_grid"]).any(0).sum()),
        water_zone_hours_lost=float(zone_down.sum()),
        person_hours_without_water=float((zone_down * A["zone_pop"][None, :]).sum()),
        critical_service_hours_lost=float(down.sum() + zone_down.sum()),
    )

In [ ]:
json.dump(dict(P=P, TAIL_H=TAIL_H), open(os.path.join(OUT, "cascade_params.json"), "w"), indent=2)   # reused by notebook 05
G = build_dependency_graph(fac)
nx.write_graphml(G, os.path.join(OUT, "dependency_graph.graphml"))
A = dependency_arrays(G)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, acyclic = {nx.is_directed_acyclic_graph(G)}")
print("Topological generations:", A["generations"])

fig, ax = plt.subplots(figsize=(10, 4.5))
posd = nx.multipartite_layout(G, subset_key="layer", align="horizontal", scale=2)
posd = {k: (v[0], -v[1]) for k, v in posd.items()}
col = {"bus": "#555555", "pump": "#2c6fbb", "water_zone": "#9ecae1", "hospital": "#c0392b"}
edge_col = ["#2c6fbb" if d["dep"] == "water" else "#bbbbbb" for _, _, d in G.edges(data=True)]
nx.draw_networkx_edges(G, posd, ax=ax, edge_color=edge_col, width=0.4, arrows=False)
for k, c in col.items():
    nl = [n for n, d in G.nodes(data=True) if d["kind"] == k]
    nx.draw_networkx_nodes(G, posd, nodelist=nl, node_color=c, node_size=12, ax=ax, label=k.replace("_", " "))
ax.legend(frameon=False, fontsize=7, loc="upper right"); ax.axis("off")
ax.set_title("Dependency graph: power (grey) and water (blue) edges, layered by topological generation")
savefig("fig02_dependency_graph.png")

## 6. Outage scenarios from notebooks 01 and 03
Each scenario gives hourly load shed per bus. It becomes a shed *fraction* per bus, and the simulation runs to `TAIL_H` hours beyond the last shed hour so that reservoirs and tanks can refill.

In [ ]:
meta = pd.read_csv(require(os.path.join(RES["01_contingency"], "scenarios_meta.csv"), "01"))
ts = pd.read_csv(os.path.join(RES["01_contingency"], "bus_shed_timeseries.csv"))
hm = os.path.join(RES["03_heat"], "heat_scenarios_meta.csv")
if os.path.exists(hm):
    meta = pd.concat([meta, pd.read_csv(hm)], ignore_index=True)
    ts = pd.concat([ts, pd.read_csv(os.path.join(RES["03_heat"], "heat_bus_shed_timeseries.csv"))], ignore_index=True)
else:
    print("No heat scenarios found (run notebook 03 to include them)")
bus_load = buses.load_mw
TS = {k: v for k, v in ts.groupby("scenario_id")}
meta.groupby("family").agg(scenarios=("scenario_id", "size"), mean_peak_shed_mw=("peak_shed_mw", "mean"),
                           mean_duration_h=("duration_h", "mean"))

In [ ]:
def run_scenario(sid, params=P, backup_fn=None):
    t = TS.get(sid, pd.DataFrame(columns=["hour", "bus", "shed_mw"]))
    horizon = int(t.hour.max() + 1 + TAIL_H) if len(t) else TAIL_H
    F = shed_fraction_matrix(t, A["bus_ids"], bus_load, horizon)
    backup = backup_fn(horizon) if backup_fn else None
    return F, cascade_simulate(A, F, params, backup)

rows, fac_rows = [], []
for _, m in meta.iterrows():
    F, R = run_scenario(m.scenario_id)
    met = cascade_metrics(A, R)
    rows.append(dict(scenario_id=m.scenario_id, family=m.family, peak_shed_mw=m.peak_shed_mw, duration_h=m.duration_h, **met))
    # longest continuous grid-off spell per facility (design outage for notebook 05)
    for kind, grid, names in [("hospital", R["hosp_grid"], A["nodes"]["hospital"]), ("pump", R["pump_grid"], A["nodes"]["pump"])]:
        off = ~grid
        if not off.any():
            continue
        for j in np.where(off.any(0))[0]:
            x = np.r_[0, off[:, j].astype(int), 0]; dx = np.diff(x)
            spells = np.where(dx == -1)[0] - np.where(dx == 1)[0]
            down = ~R["hosp_ok"][:, j] if kind == "hospital" else ~R["zone_water"][:, j]
            fac_rows.append(dict(scenario_id=m.scenario_id, family=m.family, facility=names[j], type=kind,
                                 grid_off_h=int(off[:, j].sum()), longest_spell_h=int(spells.max()), service_down_h=int(down.sum())))
cres = pd.DataFrame(rows)
cres.to_csv(os.path.join(OUT, "cascade_results.csv"), index=False)
fout = pd.DataFrame(fac_rows)
fout.to_csv(os.path.join(OUT, "facility_outages.csv"), index=False)
print(f"{len(cres)} scenarios simulated; {len(fout)} facility-scenario outages")

## 7. Results: critical service-hours lost

In [ ]:
summ = cres.groupby("family").agg(
    scenarios=("scenario_id", "size"),
    hospitals_affected=("hospitals_affected", "mean"),
    hospital_hours=("hospital_hours_lost", "mean"),
    of_which_cascade=("hospital_hours_water", "mean"),
    bed_hours=("bed_hours_lost", "mean"),
    water_zone_hours=("water_zone_hours_lost", "mean"),
    person_hours_no_water_M=("person_hours_without_water", lambda x: x.mean() / 1e6),
    critical_service_hours=("critical_service_hours_lost", "mean"),
    critical_service_hours_p95=("critical_service_hours_lost", lambda x: x.quantile(0.95)),
)
summ["cascade_share_%"] = 100 * summ.of_which_cascade / summ.hospital_hours.replace(0, np.nan)
summ.to_csv(os.path.join(OUT, "cascade_summary.csv"))
summ.round(1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
s2 = summ.sort_values("critical_service_hours")
ax[0].barh(s2.index, s2.hospital_hours - s2.of_which_cascade, color="#c0392b", label="hospital-hours: no power")
ax[0].barh(s2.index, s2.of_which_cascade, left=s2.hospital_hours - s2.of_which_cascade, color="#2c6fbb", label="hospital-hours: no water (cascade)")
ax[0].barh(s2.index, s2.water_zone_hours, left=s2.hospital_hours, color="#9ecae1", label="water-zone-hours without supply")
ax[0].set_xlabel("Critical service-hours lost (mean per event)"); ax[0].legend(frameon=False, fontsize=7)
mc = cres[cres.family.str.contains("_mc")]
for fam, d in mc.groupby("family"):
    ax[1].scatter(d.peak_shed_mw, d.critical_service_hours_lost, s=14, alpha=0.7, label=fam)
ax[1].set_xlabel("Peak load shed (MW)"); ax[1].set_ylabel("Critical service-hours lost"); ax[1].legend(frameon=False, fontsize=7)
ax[1].set_title("Storm Monte Carlo: grid impact vs service impact")
savefig("fig03_service_hours.png")

In [ ]:
# One storm event in detail: how the cascade unfolds hour by hour
sid = cres[cres.family.str.contains("extreme")].sort_values("critical_service_hours_lost").scenario_id.iloc[-1] \
      if cres.family.str.contains("extreme").any() else cres.sort_values("critical_service_hours_lost").scenario_id.iloc[-1]
F, R = run_scenario(sid)
hrs = np.arange(F.shape[0])
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.step(hrs, (~R["pump_grid"]).sum(1), where="post", label="pumping stations without grid power", color="#2c6fbb", ls="--")
ax.step(hrs, (~R["zone_water"]).sum(1), where="post", label="water zones without supply", color="#2c6fbb")
ax.step(hrs, (~R["hosp_grid"]).sum(1), where="post", label="hospitals without grid power", color="#c0392b", ls="--")
ax.step(hrs, (~R["hosp_ok"]).sum(1), where="post", label="hospitals not functional", color="#c0392b")
ax.set_xlabel("Hours after event"); ax.set_ylabel("Facilities"); ax.legend(frameon=False, fontsize=7)
ax.set_title(f"Cascade timeline: {sid}")
savefig("fig04_cascade_timeline.png")

## 8. Sensitivity to buffers
How strongly do storage and backup fuel damp the cascade? Each buffer is varied one at a time for the storm scenarios.

In [ ]:
storm_ids = meta[meta.family.str.startswith("storm")].scenario_id.tolist()
tests = {"WATER_STORAGE_H": [0, 4, 8, 16, 24], "HOSP_WATER_H": [0, 6, 12, 24], "DG_HOURS_MAJOR": [0, 8, 24, 48], "DG_HOURS_SMALL": [0, 4, 12, 24]}
sens = []
for k, vals in tests.items():
    for v in vals:
        p2 = dict(P); p2[k] = v
        tot = np.mean([cascade_metrics(A, run_scenario(s, p2)[1])["critical_service_hours_lost"] for s in storm_ids]) if storm_ids else np.nan
        sens.append(dict(parameter=k, value=v, mean_service_hours=tot))
sens = pd.DataFrame(sens)
fig, ax = plt.subplots(1, len(tests), figsize=(12, 3), sharey=True)
for a, (k, d) in zip(ax, sens.groupby("parameter", sort=False)):
    a.plot(d.value, d.mean_service_hours, marker="o"); a.axvline(P[k], color="grey", ls=":", lw=1); a.set_xlabel(f"{k} (h)")
ax[0].set_ylabel("Mean critical service-hours\nlost per storm event")
savefig("fig05_buffer_sensitivity.png")
sens.pivot(index="value", columns="parameter", values="mean_service_hours").round(0)

## 9. Notes for the technical note
- **Metric:** critical service-hours lost, split into direct (power) and cascade (water) hospital-hours, plus person-hours without water. These map directly to Sendai Target D (disruption of basic services: health, water).
- **Data:** OpenStreetMap facilities, WorldPop 2020 through Earth Engine, and grid scenarios from notebooks 01 and 03.
- **Limitations:**
  - **synthetic facility-to-bus mapping**
  - OSM completeness varies (check the counts against municipal records)
  - nearest-node water service areas
  - single-source water supply per zone
  - deterministic buffers
  - no telecom or transport dependencies
  - DG reliability not modelled (diesel sets often fail to start)

**Next:** notebook 05 (PV + battery mitigation).